In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [4]:
DATA_DIR = "../data/raw1"

data = []
labels = []

for file in os.listdir(DATA_DIR):
    if file.endswith(".csv"):
        path = os.path.join(DATA_DIR, file)
        df = pd.read_csv(path, header=None)
        data.append(df.iloc[:, :-1])
        labels.append(df.iloc[:, -1])

X = pd.concat(data).values
y = pd.concat(labels).values

X.shape, y.shape


((1000, 63), (1000,))

In [5]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

encoder.classes_


array(['B', 'HELLO', 'NO', 'THANK_YOU', 'YES'], dtype=object)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape


((800, 63), (200, 63))

In [7]:
model = Sequential([
    tf.keras.Input(shape=(63,)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(len(encoder.classes_), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               8192      
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dense_2 (Dense)             (None, 5)                 325       
                                                                 
Total params: 16,773
Trainable params: 16,773
Non-trainable params: 0
_________________________________________________________________


In [8]:
history = model.fit(
    X_train,
    y_train,
    epochs=25,
    validation_data=(X_test, y_test)
)


Epoch 1/25
25/25 [==============================] - 2s 19ms/step - loss: 1.3684 - accuracy: 0.6150 - val_loss: 1.1640 - val_accuracy: 1.0000
Epoch 2/25
25/25 [==============================] - 0s 4ms/step - loss: 0.9708 - accuracy: 0.8975 - val_loss: 0.8111 - val_accuracy: 1.0000
Epoch 3/25
25/25 [==============================] - 0s 4ms/step - loss: 0.6305 - accuracy: 1.0000 - val_loss: 0.5324 - val_accuracy: 1.0000
Epoch 4/25
25/25 [==============================] - 0s 5ms/step - loss: 0.4033 - accuracy: 1.0000 - val_loss: 0.3362 - val_accuracy: 1.0000
Epoch 5/25
25/25 [==============================] - 0s 4ms/step - loss: 0.2505 - accuracy: 1.0000 - val_loss: 0.2011 - val_accuracy: 1.0000
Epoch 6/25
25/25 [==============================] - 0s 4ms/step - loss: 0.1565 - accuracy: 1.0000 - val_loss: 0.1260 - val_accuracy: 1.0000
Epoch 7/25
25/25 [==============================] - 0s 5ms/step - loss: 0.0986 - accuracy: 1.0000 - val_loss: 0.0813 - val_accuracy: 1.0000
Epoch 8/25
25/25 [=

In [9]:
import os
import pickle

MODEL_DIR = "../model"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save trained model
model.save(os.path.join(MODEL_DIR, "gesture_model.h5"))

# Save label encoder
with open(os.path.join(MODEL_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(encoder, f)
